In [ ]:
library(Seurat)

library(reticulate)
library(anndata)

library(ggplot2)
library(ggrepel)
library(ggpubr)
library(ggpattern)
library(pheatmap)
library(dplyr)
library(tidyr)
library(RColorBrewer)
library(clustree)
library(repr)
library(patchwork)
options(repr.plot.width=10, repr.plot.height=8)


library(UpSetR)
library(grid)

library(PRROC)
library(Matrix)

getwd()

dataset_id <- "simulated_mm_RA"
genome_id <- "mm10"
samples <- c("old", "young")
colorSamples <- c("old"="#7a646a",
                "young"="#88bce7")
                
dir.create(paste0("figures_", dataset_id))         
for (sample in samples) {
    dir.create(paste0("figures_", dataset_id, "_", sample))
}

colorTools <- c( # "MATES"="#F98A7B",
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#F3E088",
    "SoloTE_unique" = "#A4DD9B",
    # "SoloTE_thr2" = "#6FC69D",
    # "SoloTE_thr1" = "#4BB2BB",
    # "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "simulated" = "grey70"
)

thrMinCells <- 500 * 0.05

In [ ]:
gc()

# Load seurat objects

In [ ]:
load("workspaces/00_evaluation_objectCreation.Rdata")

colorTools <- c( # "MATES"="#F98A7B",
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#F3E088",
    "SoloTE_unique" = "#A4DD9B",
    "SoloTE_thr2" = "#6FC69D",
    "SoloTE_thr1" = "#4BB2BB",
    "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "Stellarscope_manualrun" = "#2E3D72",
    "simulated" = "grey70"
)

In [ ]:
dataset_ids <- c("simulated_mm_RA_5kRpC", 
    "simulated_mm_RA_20kRpC", 
    "simulated_mm_RA_50kRpC")
depths <- c("5kRpC", "20kRpC", "50kRpC", "100kRpC")
names(depths) <- dataset_ids

objList_byDepth <- list()

for(id in dataset_ids){

    objList_byDepth[[as.character(depths[id])]] <- readRDS(paste0("data/objList_", id, ".RDS"))
}

objList_byDepth[["100kRpC"]] <- objList[names(objList_byDepth[["50kRpC"]])]

names(objList_byDepth)
names(objList_byDepth[["100kRpC"]])

# Correlations

### TP COUNTS

In [ ]:
library(data.table)

layer <- "data"
results_list <- list()

for (sample in samples) {

    splatter_obj <- splatter_objs[[sample]]
    splatterMat_full <- GetAssayData(splatter_obj, layer = layer)   # computed once per sample
    splatter_features <- Features(splatter_obj)

    for (depth in depths) {

        for (tool in names(objList_byDepth[[depth]])) {

            obj <- objList_byDepth[[depth]][[tool]][[sample]]

            TP_TEs <- intersect(splatter_features, Features(obj))

            splatter_subset <- as.matrix(splatterMat_full[TP_TEs, , drop = FALSE])
            tool_subset     <- as.matrix(GetAssayData(obj, layer = layer)[TP_TEs, , drop = FALSE])

            # vectorized melt
            dt <- data.table(
                TE         = rep(TP_TEs, times = ncol(splatter_subset)),
                Cell       = rep(colnames(splatter_subset), each = nrow(splatter_subset)),
                sim_count  = as.vector(splatter_subset),
                tool_count = as.vector(tool_subset)
            )

            dt <- dt[sim_count > 0 & tool_count > 0]

            dt[, order := conversion_table$class[match(TE, conversion_table$stellarscopeID)]]
            dt[, `:=`(tool = tool, sample = sample, depth = depth)]

            results_list[[paste(sample, depth, tool, sep = "_")]] <- dt
        }
    }
}

densityPlotsDf <- rbindlist(results_list)

In [ ]:
densityPlotsDf$family <- conversion_table$family[match(densityPlotsDf$TE, conversion_table$stellarscopeID)]
densityPlotsDf$subfamily <- conversion_table$subfamily[match(densityPlotsDf$TE, conversion_table$stellarscopeID)]
densityPlotsDf$propSubs <- conversion_table$propSubs[match(densityPlotsDf$TE, conversion_table$stellarscopeID)]

In [ ]:
# group by TE, depth and tool and compute spearman correlations, only for loci detected in at least 25 cells

cor_by_TE <- densityPlotsDf %>%
  group_by(TE, tool, sample, depth) %>%
  filter(n() >= thrMinCells) %>%
  summarise(
    order  = first(order),
    family = first(family),
    subfamily = first(subfamily),
    n = n(),
    cor = cor(sim_count, tool_count, method="spearman", use = "complete.obs"),
    p_value = cor.test(sim_count, tool_count, method = "spearman", use = "complete.obs")$p.value,
    .groups = "drop"
  )


In [ ]:
head(cor_by_TE)

In [ ]:
colorTools <- c( # "MATES"="#F98A7B",
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#F3E088",
    "SoloTE_unique" = "#A4DD9B",
    # "SoloTE_thr2" = "#6FC69D",
    # "SoloTE_thr1" = "#4BB2BB",
    # "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "simulated" = "grey70"
)

In [ ]:
cor_by_TE <- cor_by_TE[cor_by_TE$tool %in% names(colorTools),]

# order tools to plot
cor_by_TE$tool <- factor(cor_by_TE$tool , levels = rev(names(colorTools)))

cor_by_TE$depth <- factor(cor_by_TE$depth, levels = c("5kRpC", "20kRpC", "50kRpC", "100kRpC"))

# add other covariates
cor_by_TE$mya <- conversion_table$mya[match(cor_by_TE$TE, conversion_table$stellarscopeID)]
cor_by_TE$propSubs <- conversion_table$propSubs[match(cor_by_TE$TE, conversion_table$stellarscopeID)]
cor_by_TE$TElength <- conversion_table$end[match(cor_by_TE$TE, conversion_table$stellarscopeID)] - 
                                        conversion_table$start[match(cor_by_TE$TE, conversion_table$stellarscopeID)] 

head(cor_by_TE)



In [ ]:

options(repr.plot.width=11, repr.plot.height=12)

ggplot(cor_by_TE[cor_by_TE$sample %in% c("old","young"),]) + 
        aes(x=cor, fill=tool,  y=tool, alpha=depth) + 
  facet_wrap(~sample, ncol=2) +
  geom_boxplot( linewidth = 0.5, ) + 
  ggtitle("Correlation between estimated and", 
          subtitle = " simulated TP counts\n") +
  xlab("Spearman cor. coef.") +
  scale_fill_manual(values = colorTools) +
  theme_pubr() + 
  guides(fill="none", color="none", alpha=guide_legend(title="Depth", override.aes=list(fill="grey30"))) +
  theme(text=element_text(size=20), axis.text.y = element_text(size=20),
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        strip.text = element_text(size=20, hjust=0.5, vjust = 1 ), 
        panel.spacing = unit(3, "lines")) 
ggsave(paste0("figures_",dataset_id,"/correlation_perTE_boxplot_byAge_byTool_byDepth_TPcounts.pdf"),width=11, height=12)

In [ ]:

options(repr.plot.width=11, repr.plot.height=10)

ggplot(cor_by_TE[cor_by_TE$sample %in% c("old","young"),]) + 
        aes(x=tool, y=cor, fill=tool, alpha=depth) + 
  facet_wrap(~sample, ncol=2) +
  geom_violin(linewidth = 0.5, scale = "width", width = 0.6,
              position = position_dodge(width = 0.7)) + 
  geom_violin(aes(group = interaction(tool, depth)), 
              fill = NA, alpha = 1, color = "black",
              linewidth = 0.5, scale = "width", width = 0.6,
              draw_quantiles = c(0.5),
              position = position_dodge(width = 0.7),
              show.legend = FALSE) +
  coord_flip() +
  scale_alpha_discrete() +
  ggtitle("Correlation between estimated and", 
          subtitle = " simulated TP loci\n") +
  ylab("Spearman cor. coef.") +
  xlab("tool") +
  scale_fill_manual(values = colorTools) +
  theme_pubr() + 
  guides(fill="none", color="none", alpha=guide_legend(title="Depth", override.aes=list(fill="grey30"))) +
  theme(text=element_text(size=20), axis.text.y = element_text(size=20),
        plot.title = element_text(size=18, hjust=0.5), 
        plot.subtitle = element_text(size=18, hjust=0.5), 
        strip.text = element_text(size=20, hjust=0.5, vjust = 1 ), 
        panel.spacing = unit(3, "lines"))
ggsave(paste0("figures_",dataset_id,"/correlation_perTE_violin_byAge_byTool_byDepth_TPcounts.pdf"),width=11, height=10)